In [ ]:
!pip install fastapi uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 5.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# Создаем приложение FastAPI
app = FastAPI()

origins = [
    "https://trolling.onrender.com",
]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictRequest(BaseModel):
    text: str

@app.get("/")
def read_root():
    return {"message": "Model server is running"}

# Маршрут для анализа текста
@app.post("/predict/")
def predict(text:PredictRequest):
    toxic_prob = predict_toxicity(text.text)
    return {"prediction": toxic_prob}

In [ ]:
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Запускаем сервер в отдельном потоке
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [22864]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
!fuser -k 8000/tcp # Останавливать сервер можно этой командой

In [ ]:
# Скачиваем бинарный файл cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared

# Даем права на выполнение
!chmod +x cloudflared

# Запускаем cloudflared с привязкой к нужному порту и сохраняем вывод в файл
!nohup ./cloudflared tunnel --url http://localhost:8000 --no-autoupdate > cloudflared.log 2>&1 &

# Ожидаем пару секунд, чтобы туннель запустился
import time
time.sleep(15)

# Читаем файл с логами и выводим ссылку на туннель
!grep -o 'https://[^\"]*' cloudflared.log

cloudflared: Text file busy
https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
https://bi-discussion-gp-el.trycloudflare.com                                             |
https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.


In [ ]:
import requests

# Адрес эндпоинта
url = "https://bi-discussion-gp-el.trycloudflare.com/predict/"

# Текст для предсказания, отправляем в теле запроса как JSON
data = {
    "text": "Ну потому что буквально российские войска сосут"
}

# Отправляем POST-запрос
response = requests.post(url, json=data)

# Выводим результат
if response.status_code == 200:
    print("Response:", response.json())
else:
    print(f"Error: {response.status_code}, {response.text}")


INFO:     34.91.140.7:0 - "POST /predict/ HTTP/1.1" 200 OK
Response: {'prediction': 0.6613937031516117}


In [ ]:
data_list = []
with open("dataset1.txt", encoding='utf-8') as file:
    for line in file:
        labels = line.split()[0]
        text = line[len(labels)+1:].strip()
        labels = labels.split(",")
        mask = [0 if "__label__NORMAL" in labels else 1]
        data_list.append((text, *mask))

In [ ]:
df1 = pd.DataFrame(data_list, columns=["text", "isToxic"])

In [ ]:
df2 = pd.read_csv('labeled.csv')

In [ ]:
df2 = df2.rename(columns={'comment': 'text', 'toxic': 'isToxic'})

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
df

,text,isToxic
0,скотина! что сказать,1.0
1,я сегодня проезжала по рабочей и между домами ...,0.0
2,очередной лохотрон. зачем придумывать очередно...,0.0
3,"ретро дежавю ... сложно понять чужое сердце , ...",0.0
4,а когда мы статус агрогородка получили?,0.0
...,...,...
262697,Вонючий совковый скот прибежал и ноет. А вот и...,1.0
262698,А кого любить? Гоблина тупорылого что-ли? Или ...,1.0
262699,"Посмотрел Утомленных солнцем 2. И оказалось, ч...",0.0
262700,КРЫМОТРЕД НАРУШАЕТ ПРАВИЛА РАЗДЕЛА Т.К В НЕМ Н...,1.0


In [ ]:
X = df['text']
y = df['isToxic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, solver='liblinear')
model.fit(X_train_vectorized, y_train)

y_pred = model.predict(X_test_vectorized)

accuracy = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.99      0.97     42522
         1.0       0.94      0.78      0.86     10019

    accuracy                           0.95     52541
   macro avg       0.95      0.89      0.91     52541
weighted avg       0.95      0.95      0.95     52541



In [ ]:
def predict_toxicity(comment):
    comment_vectorized = vectorizer.transform([comment])
    prediction = model.predict(comment_vectorized)
    return 'Токсичный' if prediction[0] == 1 else 'Нетоксичный'

new_comment = "Ну потому что буквально российские войска сосут"
result = predict_toxicity(new_comment)
print(f'Комментарий: "{new_comment}" - {result}')

Комментарий: "Ну потому что буквально российские войска сосут" - Токсичный


In [ ]:
def predict_toxicity(comment):
    comment_vectorized = vectorizer.transform([comment])
    probabilities = model.predict_proba(comment_vectorized)[0]  # Получаем вероятности для обоих классов
    toxic_probability = probabilities[1]  # Вероятность токсичности
    return toxic_probability

new_comment = "Дуров, верни стену!"
toxic_prob = predict_toxicity(new_comment)
print(f'Комментарий: "{new_comment}" - Вероятность токсичности: {toxic_prob:.2f}')

Комментарий: "Дуров, верни стену!" - Вероятность токсичности: 0.17
